# LSTM-based Tagalog/Taglish NSFW Text Detection

Bidirectional LSTM binary classifier (**Safe** vs **NSFW**).

**This notebook is config-driven** — every hyperparameter lives in `config.yaml`,
so re-running on any machine reproduces the same model. It uses a proper
**70 / 15 / 15 stratified train / val / test split**: the test set is held out
and only touched for the final reported metrics.

In [ ]:
import subprocess, sys

REQUIRED = {
    'tensorflow': 'tensorflow',
    'pandas': 'pandas',
    'numpy': 'numpy',
    'scikit-learn': 'sklearn',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'pyyaml': 'yaml',
}

def install_requirements():
    for dist, mod in REQUIRED.items():
        try:
            __import__(mod)
        except ImportError:
            print(f"Installing {dist}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", dist])

print("Checking dependencies...")
install_requirements()
print("All dependencies available.")

Checking dependencies...
All dependencies available.


In [34]:
import os
import json
import pickle

import numpy as np
import pandas as pd
import yaml
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns

## Configuration

All hyperparameters are read from `config.yaml`. `configure_gpu()` only enables
memory growth if a GPU is present — it does **not** choose hyperparameters from
hardware specs (the old `HardwareConfig` did, which made results machine-dependent).

In [ ]:
class Config:

    def __init__(self, path='config.yaml'):
        with open(path, 'r') as f:
            cfg = yaml.safe_load(f)
        m, t, d, p = cfg['model'], cfg['train'], cfg['data'], cfg['paths']

        # Model architecture
        self.vocab_size = m['vocab_size']
        self.embedding_dim = m['embedding_dim']
        self.lstm_units = m['lstm_units']
        self.dense_units = m['dense_units']
        self.max_sequence_length = m['max_sequence_length']
        self.spatial_dropout = m['spatial_dropout']
        self.dropout_rate = m['dropout_rate']

        # Training
        self.test_size = t['test_size']
        self.val_size = t['val_size']
        self.batch_size = t['batch_size']
        self.epochs = t['epochs']
        self.learning_rate = t['learning_rate']
        self.random_seed = t['random_seed']
        self.use_class_weights = t['use_class_weights']
        self.monitor_metric = t['monitor_metric']
        self.threshold = t['threshold']

        # Data
        self.dataset_csv = d['dataset_csv']
        self.text_column = d['text_column']
        self.label_column = d['label_column']
        self.class_names = d['labels']

        # Paths
        self.model_dir = p['model_dir']
        self.best_model = p['best_model']
        self.final_model = p['final_model']
        self.tokenizer_path = p['tokenizer']
        self.tflite_model = p['tflite_model']
        self.metrics_path = p['metrics']

        # Reproducibility
        np.random.seed(self.random_seed)
        tf.random.set_seed(self.random_seed)

    def print_config(self):
        train_frac = 1.0 - self.val_size - self.test_size
        print("\n" + "=" * 70)
        print("CONFIGURATION (from config.yaml)")
        print("=" * 70)
        print(f"Vocab size:             {self.vocab_size}")
        print(f"Embedding dim:          {self.embedding_dim}")
        print(f"LSTM units:             {self.lstm_units}")
        print(f"Dense units:            {self.dense_units}")
        print(f"Max sequence length:    {self.max_sequence_length}")
        print(f"Dropout / spatial:      {self.dropout_rate} / {self.spatial_dropout}")
        print(f"Batch size:             {self.batch_size}")
        print(f"Epochs:                 {self.epochs}")
        print(f"Learning rate:          {self.learning_rate}")
        print(f"Split (train/val/test): {train_frac:.2f} / {self.val_size} / {self.test_size}")
        print(f"Class weights:          {self.use_class_weights}")
        print(f"Checkpoint monitor:     {self.monitor_metric}")
        print(f"Decision threshold:     {self.threshold}")
        print("=" * 70 + "\n")


def configure_gpu():
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            print(f"GPU detected: {gpus[0].name} (memory growth enabled)")
        except RuntimeError as e:
            print(f"GPU configuration error: {e}")
    else:
        print("No GPU detected - training on CPU.")

## Data loading

In [36]:
def load_dataset(config):
    print(f"Loading dataset from {config.dataset_csv}...")
    try:
        df = pd.read_csv(config.dataset_csv)
    except FileNotFoundError:
        print(f"'{config.dataset_csv}' not found - creating a tiny sample dataset for a smoke test.")
        return create_sample_dataset(config)

    print(f"Loaded. Shape: {df.shape}")
    required = [config.text_column, config.label_column]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = df.dropna(subset=required).copy()
    df[config.text_column] = df[config.text_column].astype(str).str.strip()
    df[config.label_column] = df[config.label_column].astype(int)
    df = df[df[config.text_column].str.len() > 0]

    labels = set(df[config.label_column].unique())
    if not labels.issubset({0, 1}):
        raise ValueError(f"Labels must be 0 (Safe) or 1 (NSFW); found {labels}")

    print(f"Clean shape: {df.shape}")
    print(f"Label distribution:\n{df[config.label_column].value_counts()}")
    return df

In [ ]:
def create_sample_dataset(config):
    safe = [
        'Magandang umaga sa lahat', 'Kumusta kayo ngayong araw',
        'Salamat sa inyong tulong', 'Masarap ang pagkain sa kusina',
        'Nag-aaral ako ng Filipino', 'Maligayang kaarawan',
        'Mahal kita', 'Ingat kayo palagi', 'Salamat sa Diyos', 'Mabuhay',
    ]
    nsfw = [
        'ulol manigas ka', 'putangina mo', 'tangina naman',
        'gago ka talaga', 'punyeta ka', 'bobo mo sobra',
        'leche ka', 'walang kwenta ka', 'tarantado ka', 'hayop ka',
    ]
    df = pd.DataFrame({
        config.text_column: safe + nsfw,
        config.label_column: [0] * len(safe) + [1] * len(nsfw),
    })
    df.to_csv(config.dataset_csv, index=False)
    print(f"Sample dataset written to {config.dataset_csv} ({len(df)} rows)")
    return df

## Preprocessing — stratified train / val / test split

The **test** set is carved off first and never used for tuning. The tokenizer is
fit on the **training** split only, so no vocabulary leaks from val/test.

In [ ]:
def preprocess_data(df, config):
    print("Preprocessing (stratified train/val/test split)...")
    X = df[config.text_column].values
    y = df[config.label_column].values

    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=config.test_size,
        random_state=config.random_seed, stratify=y
    )

    val_fraction = config.val_size / (1.0 - config.test_size)
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=val_fraction,
        random_state=config.random_seed, stratify=y_temp
    )

    print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

    tokenizer = Tokenizer(
        num_words=config.vocab_size,
        oov_token='<OOV>',
        filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n',
        lower=True
    )
    tokenizer.fit_on_texts(X_train)

    def encode(texts):
        seqs = tokenizer.texts_to_sequences(texts)
        return pad_sequences(seqs, maxlen=config.max_sequence_length,
                             padding='post', truncating='post')

    X_train_pad = encode(X_train)
    X_val_pad = encode(X_val)
    X_test_pad = encode(X_test)

    print(f"Learned vocabulary: {len(tokenizer.word_index) + 1} tokens")
    print(f"Sequence shape: {X_train_pad.shape}")
    return X_train_pad, X_val_pad, X_test_pad, y_train, y_val, y_test, tokenizer

## Model architecture

Stacked bidirectional LSTM. `recurrent_dropout` is intentionally **omitted** so the
fast cuDNN LSTM kernel stays active (the old version disabled it, making training
several times slower); regularization comes from `SpatialDropout1D` + `dropout`.
The deprecated `input_length` argument is removed (it errors on Keras 3).

In [ ]:
def build_lstm_model(config, vocab_size, use_cudnn='auto', summary=True):
    if summary:
        print("Building BiLSTM model...")
    model = keras.Sequential([
        layers.Embedding(input_dim=vocab_size, output_dim=config.embedding_dim,
                         name='embedding'),
        layers.SpatialDropout1D(config.spatial_dropout),

        layers.Bidirectional(layers.LSTM(
            config.lstm_units, dropout=config.dropout_rate,
            return_sequences=True, use_cudnn=use_cudnn)),
        layers.Bidirectional(layers.LSTM(
            config.lstm_units // 2, dropout=config.dropout_rate,
            use_cudnn=use_cudnn)),

        layers.Dense(config.dense_units, activation='relu'),
        layers.Dropout(config.dropout_rate),
        layers.Dense(config.dense_units // 2, activation='relu'),
        layers.Dropout(config.dropout_rate),

        layers.Dense(1, activation='sigmoid', name='output'),
    ])
    model.build(input_shape=(None, config.max_sequence_length))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config.learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy',
                 keras.metrics.Precision(name='precision'),
                 keras.metrics.Recall(name='recall')]
    )
    if summary:
        model.summary()
    return model

## Training

`EarlyStopping` and `ModelCheckpoint` use the **same** monitor metric, so the
weights restored into memory are the same ones written to `best_model.keras` —
the reported metrics now describe exactly the shipped checkpoint. Class weights
balance the Safe/NSFW loss to lift NSFW recall.

In [40]:
def train_model(model, X_train, y_train, X_val, y_val, config):
    print("\nTraining...")
    os.makedirs(config.model_dir, exist_ok=True)

    mode = 'min' if 'loss' in config.monitor_metric else 'max'
    callbacks = [
        EarlyStopping(monitor=config.monitor_metric, mode=mode, patience=5,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor=config.monitor_metric, mode=mode, factor=0.5,
                          patience=3, min_lr=1e-7, verbose=1),
        ModelCheckpoint(filepath=config.best_model, monitor=config.monitor_metric,
                        mode=mode, save_best_only=True, verbose=1),
    ]

    class_weight = None
    if config.use_class_weights:
        classes = np.array([0, 1])
        weights = compute_class_weight('balanced', classes=classes, y=y_train)
        class_weight = {int(c): float(w) for c, w in zip(classes, weights)}
        print(f"Class weights: {class_weight}")

    history = model.fit(
        X_train, y_train,
        batch_size=config.batch_size,
        epochs=config.epochs,
        validation_data=(X_val, y_val),
        class_weight=class_weight,
        callbacks=callbacks,
        verbose=1,
    )

    # restore_best_weights=True => in-memory model == best_model.keras
    model.save(config.final_model)
    print(f"Saved best_model  -> {config.best_model}")
    print(f"Saved final_model -> {config.final_model}")
    return history

## Evaluation & visualization

Metrics are computed on the **held-out test set** — data the model never saw
during training or checkpoint selection.

In [41]:
def evaluate_model(model, X_test, y_test, config):
    print("\nEvaluating on held-out test set...")
    y_prob = model.predict(X_test, verbose=0).flatten()
    y_pred = (y_prob > config.threshold).astype(int)

    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred,
                                   target_names=config.class_names, digits=4)
    cm = confusion_matrix(y_test, y_pred)

    os.makedirs(os.path.dirname(config.metrics_path), exist_ok=True)
    metrics_text = f"""LSTM Tagalog NSFW Detection Model Evaluation
{'=' * 70}

Model Architecture: Bidirectional LSTM
Task: Binary Classification (Safe vs NSFW)
Evaluated on: held-out TEST set ({len(y_test)} samples, threshold={config.threshold})

Overall Accuracy: {accuracy:.4f}

Classification Report:
{report}

Confusion Matrix:
{cm}
"""
    with open(config.metrics_path, 'w') as f:
        f.write(metrics_text)

    print(f"Test accuracy: {accuracy:.4f}")
    print(f"Metrics -> {config.metrics_path}")
    plot_confusion_matrix(cm, config)
    return accuracy, report

In [42]:
def plot_confusion_matrix(cm, config):
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=config.class_names, yticklabels=config.class_names)
    plt.title('Confusion Matrix (test set)')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    out = os.path.join(config.model_dir, 'confusion_matrix.png')
    plt.savefig(out)
    plt.close()
    print(f"Confusion matrix -> {out}")

In [43]:
def plot_training_history(history, config):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history.history['accuracy'], label='Train')
    axes[0].plot(history.history['val_accuracy'], label='Val')
    axes[0].set_title('Model Accuracy')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
    axes[0].legend(); axes[0].grid(True)
    axes[1].plot(history.history['loss'], label='Train')
    axes[1].plot(history.history['val_loss'], label='Val')
    axes[1].set_title('Model Loss')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
    axes[1].legend(); axes[1].grid(True)
    plt.tight_layout()
    out = os.path.join(config.model_dir, 'training_history.png')
    plt.savefig(out)
    plt.close()
    print(f"Training history -> {out}")

## Export (tokenizer, config.json, TFLite)

In [44]:
def save_artifacts(model, tokenizer, config):
    print("\nSaving artifacts...")
    os.makedirs(config.model_dir, exist_ok=True)

    with open(config.tokenizer_path, 'wb') as f:
        pickle.dump(tokenizer, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Tokenizer -> {config.tokenizer_path}")

    # Mirror the inference-time fields into config.json (used by the mobile app).
    config_json = {
        'max_sequence_length': config.max_sequence_length,
        'vocab_size': config.vocab_size,
        'embedding_dim': config.embedding_dim,
        'lstm_units': config.lstm_units,
        'dense_units': config.dense_units,
        'threshold': config.threshold,
        'class_names': config.class_names,
    }
    config_json_path = os.path.join(config.model_dir, 'config.json')
    with open(config_json_path, 'w') as f:
        json.dump(config_json, f, indent=4)
    print(f"Config -> {config_json_path}")

    # Reloads best_model.keras and converts on CPU (see export_to_tflite).
    export_to_tflite(config)

In [ ]:
def export_to_tflite(config):
    print("\nExporting to TensorFlow Lite (use_cudnn=False, fixed batch, builtins only)...")
    try:
        trained = keras.models.load_model(config.best_model)
        vocab_size = trained.get_layer('embedding').input_dim

        export_model = build_lstm_model(config, vocab_size, use_cudnn=False, summary=False)
        export_model.set_weights(trained.get_weights())

        # Wrap with a static batch=1 input so the LSTM TensorList shape is static.
        fixed_in = keras.Input(batch_shape=(1, config.max_sequence_length), dtype='float32')
        fixed_model = keras.Model(fixed_in, export_model(fixed_in))

        converter = tf.lite.TFLiteConverter.from_keras_model(fixed_model)
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]
        # Builtins only -> fuses to UnidirectionalSequenceLSTM, no Flex delegate needed.
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]

        tflite_model = converter.convert()
        with open(config.tflite_model, 'wb') as f:
            f.write(tflite_model)
        size_mb = os.path.getsize(config.tflite_model) / (1024 * 1024)
        print(f"TFLite -> {config.tflite_model} ({size_mb:.2f} MB)")
        return True
    except Exception as e:
        print(f"TFLite export failed: {e}")
        return False

## Inference smoke test

Loads the **shipped** `best_model.keras` and reads the threshold + max length from
`config.json`, padding with `truncating='post'` to match training exactly.

In [46]:
def test_inference(config, test_texts=None):
    if test_texts is None:
        test_texts = [
            "Magandang umaga sa lahat",
            "Kumusta ka",
            "Salamat po",
            "Good morning everyone",
            "ulol manigas ka",
        ]

    print("\nInference test (loads shipped best_model.keras)...")
    model = keras.models.load_model(config.best_model)
    with open(config.tokenizer_path, 'rb') as f:
        tokenizer = pickle.load(f)
    with open(os.path.join(config.model_dir, 'config.json'), 'r') as f:
        cfg = json.load(f)

    threshold = cfg.get('threshold', 0.5)
    class_names = cfg.get('class_names', config.class_names)
    for text in test_texts:
        seq = tokenizer.texts_to_sequences([text])
        padded = pad_sequences(seq, maxlen=cfg['max_sequence_length'],
                               padding='post', truncating='post')
        prob = float(model.predict(padded, verbose=0)[0][0])
        label = class_names[1] if prob > threshold else class_names[0]
        print(f"  '{text}' -> {label} (P(NSFW)={prob:.4f})")

## TFLite sanity check

Confirms the exported `model.tflite` reproduces `best_model.keras` on the smoke-test
phrases (within a float16 tolerance) — proof the deployed model matches the trained one.

In [ ]:
def verify_tflite(config, test_texts=None, tol=1e-2):

    if test_texts is None:
        test_texts = [
            "Magandang umaga sa lahat",
            "Kumusta ka",
            "Salamat po",
            "Good morning everyone",
            "ulol manigas ka",
        ]
    print("\nVerifying model.tflite vs best_model.keras...")

    keras_model = keras.models.load_model(config.best_model)
    with open(config.tokenizer_path, 'rb') as f:
        tokenizer = pickle.load(f)
    with open(os.path.join(config.model_dir, 'config.json'), 'r') as f:
        cfg = json.load(f)
    maxlen = cfg['max_sequence_length']

    interpreter = tf.lite.Interpreter(model_path=config.tflite_model)
    inp = interpreter.get_input_details()[0]
    out = interpreter.get_output_details()[0]
    interpreter.resize_tensor_input(inp['index'], [1, maxlen])
    interpreter.allocate_tensors()

    max_diff = 0.0
    for text in test_texts:
        seq = tokenizer.texts_to_sequences([text])
        x = pad_sequences(seq, maxlen=maxlen, padding='post',
                          truncating='post').astype('float32')

        keras_p = float(keras_model.predict(x, verbose=0)[0][0])

        interpreter.set_tensor(inp['index'], x)
        interpreter.invoke()
        tflite_p = float(interpreter.get_tensor(out['index'])[0][0])

        diff = abs(keras_p - tflite_p)
        max_diff = max(max_diff, diff)
        flag = 'OK   ' if diff <= tol else 'DIFF!'
        print(f"  [{flag}] keras={keras_p:.4f}  tflite={tflite_p:.4f}  |diff|={diff:.4f}  '{text}'")

    print(f"Max |diff| = {max_diff:.4f} (tolerance {tol})")
    print("PASS: TFLite matches the Keras model." if max_diff <= tol
          else "WARN: TFLite diverges beyond tolerance - investigate before deploying.")
    return max_diff

## Run the full pipeline

In [ ]:
def main():
    print("=" * 70)
    print("LSTM-BASED TAGALOG/TAGLISH NSFW DETECTION")
    print("=" * 70)

    config = Config('config.yaml')
    config.print_config()
    configure_gpu()

    df = load_dataset(config)
    (X_train, X_val, X_test,
     y_train, y_val, y_test, tokenizer) = preprocess_data(df, config)

    vocab_size = min(len(tokenizer.word_index) + 1, config.vocab_size)
    model = build_lstm_model(config, vocab_size)

    history = train_model(model, X_train, y_train, X_val, y_val, config)
    accuracy, report = evaluate_model(model, X_test, y_test, config)
    plot_training_history(history, config)
    save_artifacts(model, tokenizer, config)

    print("\n" + "=" * 70)
    print("TRAINING COMPLETE")
    print("=" * 70)
    print(f"Model dir:     {config.model_dir}")
    print(f"Test accuracy: {accuracy:.4f}")
    print(f"Metrics:       {config.metrics_path}")
    print("=" * 70)
    return config


if __name__ == "__main__":
    try:
        cfg = main()
        test_inference(cfg)
        verify_tflite(cfg)
    except KeyboardInterrupt:
        print("\nInterrupted by user.")
    except Exception as e:
        import traceback
        print(f"Error: {e}")
        traceback.print_exc()
    print("\nDone.")

In [48]:
import shutil
from google.colab import files
shutil.make_archive('lstm_bilstm_outputs', 'zip', 'models')   # zips models/
files.download('lstm_bilstm_outputs.zip')
files.download('metrics/lstm_metrics.txt')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>